#Initialisation

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType,StringType
from pyspark.sql.functions import trim,col

In [0]:
df=spark.table("workspace.bronze.erp_loc_a101")

#Silver transformation

##Triming

In [0]:
for filed in df.schema.fields:
    if isinstance(filed.dataType,StringType):
        df=df.withColumn(filed.name,trim(col(filed.name)))

##customerid clean up

In [0]:
df=df.withColumn(
    "CID",
    F.regexp_replace(col("CID"), "-", "")
)

##normalising the country

In [0]:


df = df.withColumn(
    "CNTRY",
    F.when(col("CNTRY") == "DE", "Germany")
     .when(col("CNTRY").isin("US", "USA"), "United States")
     .when((col("CNTRY") == "") | col("CNTRY").isNull(), "n/a")
     .otherwise(col("CNTRY"))
)

#normalisation of table name

In [0]:
Rename={
    "CID":"customer_id",
    "CNTRY":"country"
}
for old,new in Rename.items():
    df=df.withColumnRenamed(old,new)

In [0]:
df.display()

#write to silver layer

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_loc")

#sanity check

In [0]:
%sql
select * from workspace.silver.erp_loc